# Exploratory Data Analysis (EDA) - Credit Card Fraud Detection

This notebook performs exploratory data analysis on the credit card fraud detection dataset (`creditcard.csv`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Dataset

In [ ]:
possible_paths = [
    Path("data/creditcard.csv"),
    Path("ml/data/creditcard.csv"),
    Path("../ml/data/creditcard.csv"),
    Path("creditcard.csv")
]
data_path = None
for p in possible_paths:
    if p.exists():
        data_path = p
        break
if data_path is None:
    raise FileNotFoundError("Could not find creditcard.csv")
print(f"Loading dataset from: {data_path.resolve()}")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Basic Dataset Info & Missing Values

In [ ]:
print("Missing Values:")
print(df.isnull().sum().sum())
df.info()

## 3. Class Distribution (Target Imbalance)

In [ ]:
class_counts = df['Class'].value_counts()
class_pcts = df['Class'].value_counts(normalize=True) * 100

print("Class Counts:")
print(class_counts)
print("\nClass Percentages:")
print(class_pcts)

# Plot Class Distribution
plt.figure(figsize=(6, 5))
ax = sns.countplot(x='Class', data=df, hue='Class', palette='Set1', legend=False)
plt.yscale('log')
plt.title('Class Distribution (Log Scale)')
plt.ylabel('Count (Log Scale)')
for p in ax.patches:
    ax.annotate(f"{p.get_height():,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', weight='bold')
plt.show()

## 4. Transaction Amount Analysis

In [ ]:
print("Overall Amount Stats:")
print(df['Amount'].describe())
print("\nLegit (Class=0) Amount Stats:")
print(df[df['Class'] == 0]['Amount'].describe())
print("\nFraud (Class=1) Amount Stats:")
print(df[df['Class'] == 1]['Amount'].describe())

# Boxplot comparing amounts without outliers
plt.figure(figsize=(8, 6))
sns.boxplot(x='Class', y='Amount', data=df, hue='Class', palette='Set2', showfliers=False, legend=False)
plt.title('Amount Distribution by Class (Outliers Removed)')
plt.xlabel('Class (0 = Legit, 1 = Fraud)')
plt.ylabel('Amount (EUR)')
plt.show()

## 5. Transaction Time Analysis

In [ ]:
print("Legit Time Stats:")
print(df[df['Class'] == 0]['Time'].describe())
print("\nFraud Time Stats:")
print(df[df['Class'] == 1]['Time'].describe())

# Plot density distribution of Time for both classes
plt.figure(figsize=(12, 5))
sns.kdeplot(df[df['Class'] == 0]['Time'], label='Legit', fill=True, alpha=0.3)
sns.kdeplot(df[df['Class'] == 1]['Time'], label='Fraud', fill=True, alpha=0.3)
plt.title('Transaction Time Density Distribution by Class')
plt.xlabel('Time (Seconds since first transaction)')
plt.legend()
plt.show()

## 6. Feature Correlation with Class

In [ ]:
correlations = df.corr()['Class'].drop('Class').sort_values()
print("Top 5 Negatively Correlated Features:")
print(correlations.head(5))
print("\nTop 5 Positively Correlated Features:")
print(correlations.tail(5))

# Heatmap of top correlated features
plt.figure(figsize=(10, 8))
top_features = list(correlations.head(5).index) + list(correlations.tail(5).index) + ['Class']
sns.heatmap(df[top_features].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Top Correlated Features with Class')
plt.show()